# RoBERTa Toxicity Classifier + PLEX: Training Guide

What this notebook does?

Loads RoBERTa Toxicity Classifier (SkolkovoInstitute/roberta_toxicity_classifier)

Reads pre-computed SHAP impact scores from CSV

Builds/saves word-level embeddings (CLS + merged tokens)

Aligns SHAP scores to word-level embeddings

Trains a lightweight PLEX

Evaluates with stress test


📝 What gets saved

Embeddings + SHAP scores:

toxic_train_with_shap_words.pt

toxic_test_with_shap_words.pt

Each item:
{"text","label_ids","cls_embedding","word_order","word_embeddings","shap_scores","shap_scores_aligned"}

PLEX checkpoint:

plex_seismic_roberta_toxicity.pth

Stress test results (printed):

Top-k Δprob means and 95% CIs for SHAP vs PLEX


1. Load toxic_score_shap_final.csv

2. Run the below code to extract embeddings, CLS embedding, and align SHAP scores

3. If you are using Google Colab, extract 200 sentences at a time to avoid losing generated data


In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from tqdm import tqdm
import re, string
from collections import defaultdict

# ---------- helpers ----------
def normalize_token(s: str) -> str:
    s = s.lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", s).strip()

def get_cls_and_word_embs(text, tokenizer, model, device, l2norm=False, layer_idx=-1):
    """
    Returns:
      cls_embedding: [H] tensor
      word_order: list[str] - whitespace-split words
      word_embs: [W,H] tensor aligned to word_order (subwords averaged by char spans)
    """
    # Pass 1: get actual tokenized length (so we can set per-sentence max_length)
    enc0 = tokenizer(text, return_tensors="pt", truncation=False, padding=False, return_offsets_mapping=True)
    seq_len = enc0["input_ids"].shape[1]

    # Pass 2: re-tokenize with max_length=seq_len and offsets for merging
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=seq_len,
        padding=False,
        return_offsets_mapping=True,
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    offsets = enc["offset_mapping"][0].tolist()  # list[(start,end), ...]

    with torch.no_grad():
        out = model(
            **{k: v for k, v in enc.items() if k in ["input_ids", "attention_mask"]},
            output_hidden_states=True
        )
        # choose layer (last by default)
        hidden = out.hidden_states[layer_idx].squeeze(0)  # [T,H]
    if l2norm:
        hidden = F.normalize(hidden, dim=-1)

    # CLS
    cls_embedding = hidden[0].detach().cpu()

    # Collect subword spans (skip [CLS]/[SEP] and zero-length offsets)
    sub_spans = []
    for i, (s, e) in enumerate(offsets):
        if i == 0 or i == len(offsets) - 1:
            continue
        if s == e:
            continue
        sub_spans.append(((s, e), hidden[i].detach().cpu()))

    # Word order (whitespace split) + char spans
    word_order = text.split()
    word_emb_list = []
    words_char_spans = []
    char_ptr = 0
    for w in word_order:
        w_start = text.find(w, char_ptr)
        if w_start == -1:
            w_start = char_ptr  # fallback
        w_end = w_start + len(w)
        words_char_spans.append((w_start, w_end, w))
        char_ptr = w_end

    for (w_start, w_end, w) in words_char_spans:
        # subwords overlapping the word's char span
        embs = [emb for (s, e), emb in sub_spans if not (e <= w_start or s >= w_end)]
        if len(embs) == 0:
            # relaxed fallback: subword text is substring of the word or vice-versa
            embs = [emb for (s, e), emb in sub_spans if (text[s:e] in w) or (w in text[s:e])]
        if len(embs) > 0:
            embs = torch.stack(embs, dim=0)
            word_emb = embs.mean(dim=0)
            word_emb_list.append(word_emb)
        else:
            # skip words that have no aligned subwords (e.g., pure punctuation)
            pass

    word_embs = torch.stack(word_emb_list, dim=0) if len(word_emb_list) > 0 else torch.empty(0, hidden.shape[-1])
    return cls_embedding, word_order, word_embs

# ---------- main ----------
# Load CSV data
csv_path = "../data/toxic_score_lime_final.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} rows from CSV")
print(f"Unique sentences: {df['id'].nunique()}")

# Process data by sentence ID - process all 300 sentences
unique_ids = df['id'].unique()
batch_size = 200  # Process all 300 sentences
start_idx = 0
end_idx = min(start_idx + batch_size, len(unique_ids))
batch_ids = unique_ids[start_idx:end_idx]

print(f"Processing sentences {start_idx} to {end_idx} (batch size: {len(batch_ids)})")

# Model and tokenizer
model_name = "SkolkovoInstitute/roberta_toxicity_classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")
print(f"Model hidden size: {model.config.hidden_size}")

results = []
for sent_id in tqdm(batch_ids, desc="Processing sentences"):
    # Get all rows for this sentence
    sent_df = df[df['id'] == sent_id].sort_values('token_index')
    
    if len(sent_df) == 0:
        continue
    
    # Get sentence text (should be the same for all rows)
    text = str(sent_df.iloc[0]['sentence'])
    
    # Get CLS + merged word embeddings (last layer by default)
    cls_embedding, word_order, word_embs = get_cls_and_word_embs(
        text, tokenizer, model, device, l2norm=False, layer_idx=-1
    )
    
    if len(word_order) == 0 or word_embs.shape[0] == 0:
        continue
    
    # Load SHAP scores from CSV
    shap_dict = {}
    for _, row in sent_df.iterrows():
        tok_idx = int(row['token_index'])
        impact_score = float(row['impact_score'])
        shap_dict[tok_idx] = impact_score
    
    # Align SHAP scores to word_order (whitespace words)
    # Map token_index to word positions
    aligned = [0.0 for _ in word_order]
    shap_scores = []
    shap_indices = []
    
    # Simple alignment: token_index should match word position (0-indexed)
    for tok_idx, score in shap_dict.items():
        if 0 <= tok_idx < len(word_order):
            aligned[tok_idx] = score
            shap_scores.append(score)
            shap_indices.append(tok_idx)
    
    # Normalize SHAP scores per sentence by max|score|
    if len(shap_scores) > 0:
        max_abs = max(abs(s) for s in shap_scores)
        if max_abs > 0:
            aligned = [s / max_abs for s in aligned]
    
    shap_scores_aligned = torch.tensor(aligned, dtype=torch.float32)
    
    # For label, use model's prediction (binary classification: 0 or 1)
    with torch.no_grad():
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding=False).to(device)
        logits = model(**enc).logits
        if logits.shape[-1] == 1:
            # Binary classification
            prob = torch.sigmoid(logits[0, 0]).item()
            label_id = 1 if prob > 0.5 else 0
        else:
            # Multi-class
            label_id = int(torch.argmax(logits[0]).item())
    
    # Save
    results.append({
        "text": text,
        "label_ids": [label_id],  # list for consistency
        "cls_embedding": cls_embedding.cpu(),   # [H]
        "word_order": word_order,               # whitespace words
        "word_embeddings": word_embs.cpu(),     # [W,H] merged
        "shap_scores": torch.tensor(shap_scores, dtype=torch.float32) if shap_scores else torch.tensor([], dtype=torch.float32),
        "shap_scores_aligned": shap_scores_aligned,  # [W] aligned to word_order
    })

# Save
output_file = f"toxic_train_with_shap_words_{start_idx}_{end_idx}.pt"
torch.save(results, output_file)
print(f"\n✅ Saved merged word embeddings + SHAP to {output_file}")
print(f"Total samples saved: {len(results)}")

# Quick peek
if results:
    ex = results[0]
    print(f"\nExample text: {ex['text'][:100]}...")
    print(f"Words (first 12): {ex['word_order'][:12]}")
    print(f"Word embeddings shape: {tuple(ex['word_embeddings'].shape)}")
    print(f"SHAP aligned (first 12): {ex['shap_scores_aligned'][:12]}")


/u/hanqim2/.conda/envs/OM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 17345 rows from CSV
Unique sentences: 300
Processing sentences 0 to 200 (batch size: 200)


Some weights of the model checkpoint at SkolkovoInstitute/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Using device: cpu
Model hidden size: 768


Processing sentences: 100%|██████████| 200/200 [00:14<00:00, 13.80it/s]



✅ Saved merged word embeddings + SHAP to toxic_train_with_shap_words_0_200.pt
Total samples saved: 200

Example text: I can literally picture you shouting GASLIGHTING at people and running away when you have no argumen...
Words (first 12): ['I', 'can', 'literally', 'picture', 'you', 'shouting', 'GASLIGHTING', 'at', 'people', 'and', 'running', 'away']
Word embeddings shape: (44, 768)
SHAP aligned (first 12): tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.6667, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000])


1. Here you can combine your datasets if you have generated 200 sentences at a time

2. To run the below code, just upload the .pt files


In [2]:
import torch, random
import os

# --- paths ---
# Add your .pt file paths here (modify as needed)
# You can specify multiple files or just one if you haven't split the data
FILE_PATHS = [
    "toxic_train_with_shap_words_0_200.pt",       # first 200
    # "toxic_train_with_shap_words_200_400.pt",   # uncomment if you have more batches
    # Add more paths as needed
]

OUT = "toxic_shap_words_train.pt"

# --- load (PyTorch 2.6: allow full unpickle) ---
all_data = []
for file_path in FILE_PATHS:
    if os.path.exists(file_path):
        print(f"Loading {file_path}...")
        data = torch.load(file_path, weights_only=False)
        all_data.append(data)
        print(f"  ✅ Loaded {len(data)} samples")
    else:
        print(f"  ⚠️  File not found: {file_path}, skipping...")

if len(all_data) == 0:
    raise FileNotFoundError("No valid .pt files found! Please check your file paths.")

print(f"\nTotal files loaded: {len(all_data)}")
print(f"Total samples before merge: {sum(len(d) for d in all_data)}")

# --- merge + optional de-dup by 'text' ---
merged = []
seen = set()
# Flatten all_data into a single list
all_samples = []
for data in all_data:
    all_samples.extend(data)

for ex in all_samples:
    txt = ex.get("text", None)
    if not txt or txt in seen:
        continue
    # light validation (skip empty/invalid samples)
    we = ex.get("word_embeddings", None)
    sa = ex.get("shap_scores_aligned", None)
    if we is None or sa is None:
        continue
    if getattr(we, "numel", lambda:0)() == 0 or getattr(sa, "numel", lambda:0)() == 0:
        continue
    # (optional) enforce matching length with word_order
    words = ex.get("word_order", [])
    if len(words) != int(we.shape[0]) or len(words) != int(sa.shape[0]):
        # skip inconsistent samples
        continue

    merged.append(ex)
    seen.add(txt)

print(f"Merged (after de-dup + validate): {len(merged)} samples")

# --- shuffle for training ---
random.seed(42)
random.shuffle(merged)

# --- save ---
torch.save(merged, OUT)
print(f"✅ Saved merged training set → {OUT}")


Loading toxic_train_with_shap_words_0_200.pt...
  ✅ Loaded 200 samples

Total files loaded: 1
Total samples before merge: 200
Merged (after de-dup + validate): 200 samples
✅ Saved merged training set → toxic_shap_words_train.pt


1. Below you can start training PLEX algorithm


In [3]:
# train_plex_from_shap.py
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.stats import spearmanr

# ---------------------------
# Config
# ---------------------------
DATA_PATH = "toxic_shap_words_train.pt"  # produced earlier
SAVE_CKPT = "plex_seismic_roberta_toxicity.pth"

# RoBERTa-base hidden size is 768
INPUT_SIZE = 768           
BATCH_SIZE = 2048          # pairs (CLS, word) per batch (adjust for your GPU/CPU)
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------
# Model (shared Siamese head)
# ---------------------------
class SeismicNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class PLEXHeadShared(nn.Module):
    """
    Projects CLS and word embeddings with shared SeismicNet,
    then computes cosine similarity per word.
    """
    def __init__(self, input_size=768):
        super().__init__()
        self.net = SeismicNet(input_size)

    def forward(self, cls_emb: torch.Tensor, word_emb: torch.Tensor) -> torch.Tensor:
        """
        cls_emb: [B, H]
        word_emb: [B, H]
        returns: [B] cosine similarities in [-1, 1]
        """
        cls_proj = self.net(cls_emb)      # [B, 64]
        word_proj = self.net(word_emb)    # [B, 64]
        word_proj = word_proj - word_proj.mean(dim=0, keepdim=True)
        cls_norm = F.normalize(cls_proj, dim=-1)
        word_norm = F.normalize(word_proj, dim=-1)
        return torch.sum(cls_norm * word_norm, dim=-1)

# ---------------------------
# Data utilities
# ---------------------------
def load_data(path):
    # weights_only=False because file contains Python objects (list of dicts)
    return torch.load(path, weights_only=False)

def split_indices(n, val_ratio=0.2, seed=SEED):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    v = int(round(n * val_ratio))
    return idx[v:], idx[:v]  # train_idx, val_idx

class PLEXPairsDataset(Dataset):
    """
    Flattens per-sentence pairs (CLS, word_i, target_i) across all sentences.
    Targets are normalized per sentence by max|target|.
    """
    def __init__(self, samples):
        self.cls = []       # [N, H]
        self.wrd = []       # [N, H]
        self.tgt = []       # [N]
        self.sent_ptrs = [] # list of (start, end) indices in flat arrays
        cur = 0
        for ex in samples:
            word_embs: torch.Tensor = ex["word_embeddings"]   # [W, H]
            shap_aligned: torch.Tensor = ex["shap_scores_aligned"]  # [W]
            cls_emb: torch.Tensor = ex["cls_embedding"]       # [H]

            if word_embs.numel() == 0 or shap_aligned.numel() == 0:
                self.sent_ptrs.append((cur, cur))
                continue

            # Normalize targets per sentence by max|score| to [-1,1]
            y = shap_aligned.clone().float()
            m = torch.max(torch.abs(y))
            if m > 0:
                y = y / m

            # Threshold to suppress tail noise
            tau = 0.05
            keep = torch.abs(y) >= tau
            if keep.any():
                word_embs = word_embs[keep]
                y = y[keep]
            else:
                # if all tiny, keep the largest 1 token to avoid empty sentence
                j = torch.argmax(torch.abs(y))
                word_embs = word_embs[j:j+1]
                y = y[j:j+1]

            W = word_embs.shape[0]
            # Repeat CLS per word
            cls_rep = cls_emb.unsqueeze(0).repeat(W, 1)       # [W, H]

            self.cls.append(cls_rep)
            self.wrd.append(word_embs.float())
            self.tgt.append(y)

            cur_next = cur + W
            self.sent_ptrs.append((cur, cur_next))
            cur = cur_next

        if len(self.cls) > 0:
            self.cls = torch.cat(self.cls, dim=0)  # [N, H]
            self.wrd = torch.cat(self.wrd, dim=0)  # [N, H]
            self.tgt = torch.cat(self.tgt, dim=0)  # [N]
        else:
            self.cls = torch.empty(0, INPUT_SIZE)
            self.wrd = torch.empty(0, INPUT_SIZE)
            self.tgt = torch.empty(0)

    def __len__(self):
        return self.tgt.shape[0]

    def __getitem__(self, idx):
        return self.cls[idx], self.wrd[idx], self.tgt[idx]

# ---------------------------
# Metrics
# ---------------------------
def evaluate_sentence_level(model, samples):
    """
    Compute per-sentence Spearman (median) and Top-k overlap (k=1,3,5) means.
    """
    model.eval()
    spearmans = []
    top1 = []; top3 = []; top5 = []

    @torch.no_grad()
    def predict_scores(cls_emb, word_embs):
        B = word_embs.shape[0]
        cls_rep = cls_emb.unsqueeze(0).repeat(B, 1).to(DEVICE)
        word_embs = word_embs.to(DEVICE)
        scores = model(cls_rep, word_embs).cpu().numpy()
        return scores

    for ex in samples:
        words = ex["word_order"]
        word_embs: torch.Tensor = ex["word_embeddings"]
        shap = ex["shap_scores_aligned"].numpy() if len(words)>0 else np.array([])
        if word_embs.numel() == 0 or shap.size == 0:
            continue

        # Normalize SHAP per sentence as in training
        if np.max(np.abs(shap)) > 0:
            shap_n = shap / np.max(np.abs(shap))
        else:
            shap_n = shap

        cls = ex["cls_embedding"]
        pred = predict_scores(cls, word_embs)

        # Spearman across all words
        if pred.size > 1 and shap_n.size > 1:
            rho, _ = spearmanr(pred, shap_n)
            if not np.isnan(rho):
                spearmans.append(rho)

        # Top-k overlap helper
        def topk_overlap(a, b, k):
            if len(a) == 0 or len(b) == 0 or len(a) != len(b) or len(a) < k:
                return np.nan
            ai = np.argsort(-a)[:k]
            bi = np.argsort(-b)[:k]
            return len(set(ai) & set(bi)) / float(k)

        top1.append(topk_overlap(pred, shap_n, 1))
        top3.append(topk_overlap(pred, shap_n, 3))
        top5.append(topk_overlap(pred, shap_n, 5))

    # Aggregate
    def nanmean(x): return float(np.nanmean(x)) if len(x)>0 else float("nan")
    def nanmedian(x): return float(np.nanmedian(x)) if len(x)>0 else float("nan")

    return {
        "spearman_median": nanmedian(spearmans),
        "top1_mean": nanmean(top1),
        "top3_mean": nanmean(top3),
        "top5_mean": nanmean(top5),
        "n_sentences": len(samples)
    }

# ---------------------------
# Training
# ---------------------------
def main():
    print("Loading data:", DATA_PATH)
    all_samples = load_data(DATA_PATH)
    # Filter out empty samples just in case
    all_samples = [ex for ex in all_samples if ex["word_embeddings"].numel() > 0 and ex["shap_scores_aligned"].numel() > 0]
    print(f"Total usable sentences: {len(all_samples)}")

    # Split sentences
    train_idx, val_idx = split_indices(len(all_samples), val_ratio=VAL_SPLIT, seed=SEED)
    train_samples = [all_samples[i] for i in train_idx]
    val_samples = [all_samples[i] for i in val_idx]
    print(f"Train sentences: {len(train_samples)} | Val sentences: {len(val_samples)}")

    # Build flat pair dataset
    train_ds = PLEXPairsDataset(train_samples)
    val_ds   = PLEXPairsDataset(val_samples)
    print(f"Train pairs: {len(train_ds)} | Val pairs: {len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    # Model, opt, loss
    model = PLEXHeadShared(input_size=INPUT_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    huber = nn.SmoothL1Loss(reduction="none")  # we'll weight it

    # Training loop
    best_val = float("inf")
    for epoch in range(1, EPOCHS+1):
        model.train()
        running = 0.0
        count = 0

        for cls_b, wrd_b, tgt_b in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            cls_b = cls_b.to(DEVICE)
            wrd_b = wrd_b.to(DEVICE)
            tgt_b = tgt_b.to(DEVICE)           # in [-1,1] (per-sentence normalized)

            pred = model(cls_b, wrd_b)         # [-1,1]
            loss_vec = huber(pred, tgt_b)

            # weight by |target| to emphasize salient tokens (avoid all zeros dominating)
            weights = torch.clamp_min(torch.abs(tgt_b), 0.1)  # floor to keep gradient on small targets
            loss = (loss_vec * weights).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            running += loss.item() * cls_b.size(0)
            count   += cls_b.size(0)

        train_loss = running / max(1, count)

        # Quick pair-level val loss
        model.eval()
        val_running = 0.0
        val_count = 0
        with torch.no_grad():
            for cls_b, wrd_b, tgt_b in val_loader:
                cls_b = cls_b.to(DEVICE)
                wrd_b = wrd_b.to(DEVICE)
                tgt_b = tgt_b.to(DEVICE)
                pred = model(cls_b, wrd_b)
                loss_vec = huber(pred, tgt_b)

                weights = torch.clamp_min(torch.abs(tgt_b), 0.1)
                loss = (loss_vec * weights).mean()
                val_running += loss.item() * cls_b.size(0)
                val_count   += cls_b.size(0)
        val_loss = val_running / max(1, val_count)

        # Sentence-level metrics (more meaningful)
        sent_metrics = evaluate_sentence_level(model, val_samples)

        print(f"\nEpoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"Spearman_median={sent_metrics['spearman_median']:.3f}  "
              f"Top1={sent_metrics['top1_mean']:.3f}  Top3={sent_metrics['top3_mean']:.3f}  "
              f"Top5={sent_metrics['top5_mean']:.3f}  n_val_sent={sent_metrics['n_sentences']}")

        # Save best by val_loss
        if val_loss < best_val:
            best_val = val_loss
            state = {
                "state_dict": model.state_dict(),
                "config": {
                    "input_size": INPUT_SIZE,
                    "epochs": EPOCHS,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "seed": SEED
                }
            }
            torch.save(state, SAVE_CKPT)
            print(f"  ✅ Saved best checkpoint -> {SAVE_CKPT}")

    print("\nDone.")

if __name__ == "__main__":
    main()


Loading data: toxic_shap_words_train.pt
Total usable sentences: 200
Train sentences: 160 | Val sentences: 40
Train pairs: 800 | Val pairs: 200


Epoch 1/50: 100%|██████████| 1/1 [00:00<00:00, 48.80it/s]



Epoch 1: train_loss=0.3314  val_loss=0.3138  Spearman_median=0.020  Top1=0.050  Top3=0.142  Top5=0.185  n_val_sent=40
  ✅ Saved best checkpoint -> plex_seismic_roberta_toxicity.pth


Epoch 2/50: 100%|██████████| 1/1 [00:00<00:00, 52.61it/s]



Epoch 2: train_loss=0.2966  val_loss=0.2052  Spearman_median=0.055  Top1=0.075  Top3=0.133  Top5=0.205  n_val_sent=40
  ✅ Saved best checkpoint -> plex_seismic_roberta_toxicity.pth


...


Epoch 48: train_loss=0.1071  val_loss=0.0837  Spearman_median=0.152  Top1=0.100  Top3=0.183  Top5=0.270  n_val_sent=40
  ✅ Saved best checkpoint -> plex_seismic_roberta_toxicity.pth


Epoch 49/50: 100%|██████████| 1/1 [00:00<00:00, 61.48it/s]



Epoch 49: train_loss=0.1056  val_loss=0.0825  Spearman_median=0.155  Top1=0.100  Top3=0.183  Top5=0.270  n_val_sent=40
  ✅ Saved best checkpoint -> plex_seismic_roberta_toxicity.pth


Epoch 50/50: 100%|██████████| 1/1 [00:00<00:00, 66.80it/s]



Epoch 50: train_loss=0.1043  val_loss=0.0820  Spearman_median=0.155  Top1=0.050  Top3=0.200  Top5=0.280  n_val_sent=40
  ✅ Saved best checkpoint -> plex_seismic_roberta_toxicity.pth

Done.


# 🧪 Interpreting Metrics

Spearman (median): Rank agreement between PLEX and SHAP across words.

Top-k overlap: Fraction of overlap in top-k important words.

Stress test (Δprob): Drop in RoBERTa's predicted probability after removing top-k words proposed by SHAP/PLEX. Higher is better. Comparable SHAP/PLEX means (overlapping CIs) ⇒ faithfulness at similar impact, with PLEX being much cheaper at inference.

Below you use the train model to test the test data

**Note:** If you don't have a separate test file, you can:
1. Use the training data (split it for testing)
2. Generate test data from CSV (uncomment the data generation section below)


In [4]:
# stress_test_prob_drop.py
import re, math
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# --------------------------
# Config
# --------------------------
# Option 1: Use existing test .pt file (if available)
TEST_PT_PATH = "toxic_test_with_shap_words.pt"

# Option 2: Use training data (for testing purposes) - RECOMMENDED if you don't have test data
TRAIN_PT_PATH = "toxic_shap_words_train.pt"  # or "toxic_train_with_shap_words_0_200.pt"
USE_TRAIN_FOR_TEST = True  # Set to True to use training data for testing (last 20%)

# Option 3: Generate test data from CSV (needs embedding extraction)
CSV_PATH = "../data/toxic_score_shap_final.csv"
GENERATE_FROM_CSV = False  # Set to True if you want to generate from CSV
TEST_CSV_START_IDX = 200  # Use sentences after index 200 as test set
TEST_CSV_END_IDX = 300  # Adjust based on your data

PLEX_CKPT = "plex_seismic_roberta_toxicity.pth"
MODEL_NAME = "SkolkovoInstitute/roberta_toxicity_classifier"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_LIST = [1, 3, 5]
SEED = 42
torch.manual_seed(SEED)

# --------------------------
# Load or generate test data
# --------------------------
if os.path.exists(TEST_PT_PATH):
    print(f"Loading test data from {TEST_PT_PATH}...")
    data = torch.load(TEST_PT_PATH, weights_only=False)
    print(f"✅ Loaded {len(data)} samples from file.")
elif USE_TRAIN_FOR_TEST and os.path.exists(TRAIN_PT_PATH) and not GENERATE_FROM_CSV:
    print(f"Using training data from {TRAIN_PT_PATH} for testing...")
    all_data = torch.load(TRAIN_PT_PATH, weights_only=False)
    # Use a subset for testing (e.g., last 20%)
    test_size = max(1, min(int(len(all_data) * 0.2), len(all_data)))
    data = all_data[-test_size:]
    print(f"✅ Using {len(data)} samples (last {test_size}) from training data for testing.")
    print(f"   (Total training samples: {len(all_data)})")
elif USE_TRAIN_FOR_TEST and os.path.exists("toxic_train_with_shap_words_0_200.pt") and not GENERATE_FROM_CSV:
    # Fallback: try the generated file directly
    print(f"Using generated training data for testing...")
    all_data = torch.load("toxic_train_with_shap_words_0_200.pt", weights_only=False)
    test_size = max(1, min(int(len(all_data) * 0.2), len(all_data)))
    data = all_data[-test_size:]
    print(f"✅ Using {len(data)} samples from generated data for testing.")
elif GENERATE_FROM_CSV and os.path.exists(CSV_PATH):
    print(f"Generating test data from CSV: {CSV_PATH}...")
    print(f"Using sentences {TEST_CSV_START_IDX} to {TEST_CSV_END_IDX}")
    
    # Import helper function from first cell (you may need to run it first)
    # For now, we'll define a simplified version here
    df = pd.read_csv(CSV_PATH)
    unique_ids = df['id'].unique()
    test_ids = unique_ids[TEST_CSV_START_IDX:TEST_CSV_END_IDX]
    
    # Load model for embeddings
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.eval()
    model.to(DEVICE)
    
    # Simple embedding extraction (simplified version)
    def get_simple_embeddings(text, tokenizer, model, device):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, 
                       return_offsets_mapping=True, padding=False).to(device)
        with torch.no_grad():
            out = model(**{k: v for k, v in enc.items() if k in ["input_ids", "attention_mask"]},
                       output_hidden_states=True)
            hidden = out.hidden_states[-1].squeeze(0)
        cls_emb = hidden[0].detach().cpu()
        word_order = text.split()
        return cls_emb, word_order
    
    data = []
    for sent_id in tqdm(test_ids, desc="Generating test data"):
        sent_df = df[df['id'] == sent_id].sort_values('token_index')
        if len(sent_df) == 0:
            continue
        text = str(sent_df.iloc[0]['sentence'])
        cls_emb, word_order = get_simple_embeddings(text, tokenizer, model, DEVICE)
        
        # Get SHAP scores
        shap_dict = {int(row['token_index']): float(row['impact_score']) 
                     for _, row in sent_df.iterrows()}
        aligned = [shap_dict.get(i, 0.0) for i in range(len(word_order))]
        
        # Normalize
        max_abs = max(abs(s) for s in aligned) if aligned else 1.0
        if max_abs > 0:
            aligned = [s / max_abs for s in aligned]
        
        data.append({
            "text": text,
            "cls_embedding": cls_emb,
            "word_order": word_order,
            "word_embeddings": cls_emb.unsqueeze(0).repeat(len(word_order), 1),  # Simplified
            "shap_scores_aligned": torch.tensor(aligned, dtype=torch.float32)
        })
    
    print(f"✅ Generated {len(data)} test samples from CSV.")
else:
    raise FileNotFoundError(
        f"No test data found! Please:\n"
        f"1. Generate test data by running the first cell with different indices, OR\n"
        f"2. Set GENERATE_FROM_CSV=True and provide CSV path, OR\n"
        f"3. Use training data by setting TRAIN_PT_PATH"
    )

# --------------------------
# RoBERTa toxicity classifier
# --------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
clf = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
clf.eval()

@torch.no_grad()
def pred_prob(text: str, target_idx: int = 0) -> float:
    enc = tok(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(DEVICE)
    logits = clf(**enc).logits
    if logits.shape[-1] == 1:
        # Binary classification
        prob = torch.sigmoid(logits[0, 0]).item()
        return prob if target_idx == 1 else (1.0 - prob)
    else:
        # Multi-class
        probs = torch.softmax(logits[0], dim=-1)
        return float(probs[target_idx].item())

@torch.no_grad()
def top_pred_idx(text: str) -> int:
    enc = tok(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(DEVICE)
    logits = clf(**enc).logits
    if logits.shape[-1] == 1:
        # Binary: return 1 if prob > 0.5
        prob = torch.sigmoid(logits[0, 0]).item()
        return 1 if prob > 0.5 else 0
    else:
        # Multi-class
        return int(torch.argmax(logits[0]).item())

# --------------------------
# PLEX head (SeismicNet shared)
# --------------------------
class SeismicNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(p=0.5)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class PLEXHeadShared(nn.Module):
    def __init__(self, input_size=768):
        super().__init__()
        self.net = SeismicNet(input_size)
    def forward(self, cls_emb: torch.Tensor, word_embs: torch.Tensor) -> torch.Tensor:
        """
        cls_emb: [W,H] or [H] (we'll expand if needed)
        word_embs: [W,H]
        returns: [W] cosine sims
        """
        if cls_emb.dim() == 1:
            # repeat cls per word
            cls_emb = cls_emb.unsqueeze(0).repeat(word_embs.shape[0], 1)
        cls_proj  = self.net(cls_emb)        # [W,64]
        word_proj = self.net(word_embs)      # [W,64]
        # Center token projections (helps dispersion)
        word_proj = word_proj - word_proj.mean(dim=0, keepdim=True)
        cls_norm  = F.normalize(cls_proj, dim=-1)
        word_norm = F.normalize(word_proj, dim=-1)
        sims = (cls_norm * word_norm).sum(dim=-1)  # [W]
        return sims

# load PLEX weights
plex = PLEXHeadShared(input_size=768).to(DEVICE)
state = torch.load(PLEX_CKPT, map_location="cpu")
sd = state.get("state_dict", state)
# If saved as bare SeismicNet keys, wrap:
if isinstance(sd, dict) and all(k.split(".")[0] in {"net","fc1","fc2"} for k in sd.keys()):
    sd = { (f"net.{k}" if k.startswith(("fc1","fc2")) else k): v for k,v in sd.items() }
missing, unexpected = plex.load_state_dict(sd, strict=False)
print("PLEX load -> missing:", missing)
print("PLEX load -> unexpected:", unexpected)
plex.eval()

@torch.no_grad()
def plex_scores_for_entry(entry) -> np.ndarray:
    we: torch.Tensor = entry["word_embeddings"]   # [W,H]
    if we.numel() == 0:
        return np.array([])
    cls: torch.Tensor = entry["cls_embedding"]    # [H]
    scores = plex(cls.to(DEVICE), we.to(DEVICE)).detach().cpu().numpy()
    return scores

# --------------------------
# Utils: remove top-k words (whitespace tokens)
# --------------------------
def remove_words(text: str, words: list[str], top_indices: np.ndarray, k: int) -> str:
    ws = words[:]  # already whitespace-split used in extraction
    pick = set(top_indices[:min(k, len(top_indices))].tolist())
    kept = [w for i, w in enumerate(ws) if i not in pick]
    out = " ".join(kept)
    out = re.sub(r"\s+", " ", out).strip()
    # avoid empty string: fall back to original if we removed everything
    return out if out else text

def topk_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if scores.size == 0:
        return np.array([], dtype=int)
    return np.argsort(-scores)[:min(k, scores.size)]

# --------------------------
# Evaluate probability drop
# --------------------------
def mean_ci(x):
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan, (np.nan, np.nan)
    m = float(np.mean(x))
    se = float(np.std(x, ddof=1) / math.sqrt(len(x))) if len(x) > 1 else np.nan
    return m, (m - 1.96*se, m + 1.96*se) if not np.isnan(se) else (np.nan, np.nan)

results = {k: {"shap": [], "plex": []} for k in K_LIST}
n_used = 0

for ex in data:
    words = ex.get("word_order", [])
    if not words:
        continue
    shap = ex.get("shap_scores_aligned", torch.tensor([]))
    if shap.numel() == 0:
        continue
    shap_np = shap.numpy()

    # PLEX scores for this entry
    plex_np = plex_scores_for_entry(ex)
    if plex_np.size == 0:
        continue

    text = ex["text"]

    # use model's own top predicted class on ORIGINAL sentence
    try:
        tgt_idx = top_pred_idx(text)
        p0 = pred_prob(text, tgt_idx)
    except Exception:
        continue

    for k in K_LIST:
        # SHAP top-k removal
        si = topk_indices(shap_np, k)
        text_s = remove_words(text, words, si, k)
        p_s = pred_prob(text_s, tgt_idx)
        results[k]["shap"].append(p0 - p_s)

        # PLEX top-k removal
        pi = topk_indices(plex_np, k)
        text_p = remove_words(text, words, pi, k)
        p_p = pred_prob(text_p, tgt_idx)
        results[k]["plex"].append(p0 - p_p)

    n_used += 1

print(f"\nEvaluated {n_used} sentences.")

# Report means + 95% CI
for k in K_LIST:
    m_s, ci_s = mean_ci(results[k]["shap"])
    m_p, ci_p = mean_ci(results[k]["plex"])
    print(f"\nTop-{k} removal Δprob (mean [95% CI]):")
    print(f"  SHAP: {m_s:.4f}  [{ci_s[0]:.4f}, {ci_s[1]:.4f}]")
    print(f"  PLEX: {m_p:.4f}  [{ci_p[0]:.4f}, {ci_p[1]:.4f}]")


Using training data from toxic_shap_words_train.pt for testing...


✅ Using 40 samples (last 40) from training data for testing.
   (Total training samples: 200)


Some weights of the model checkpoint at SkolkovoInstitute/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


PLEX load -> missing: []
PLEX load -> unexpected: []

Evaluated 40 sentences.

Top-1 removal Δprob (mean [95% CI]):
  SHAP: 0.0453  [-0.0172, 0.1079]
  PLEX: -0.0031  [-0.0082, 0.0021]

Top-3 removal Δprob (mean [95% CI]):
  SHAP: 0.0942  [0.0045, 0.1840]
  PLEX: -0.0037  [-0.0096, 0.0022]

Top-5 removal Δprob (mean [95% CI]):
  SHAP: 0.0943  [0.0045, 0.1840]
  PLEX: 0.0199  [-0.0207, 0.0604]
